# Laboratorium 2: reguła Bayesa i metoda k najbliższych sąsiadów

Karta laboratorium: <https://rodakt.github.io/MAD/laby/02-knn.html>

Treść zadań jest w książce, tutaj jest tylko szkielet kodu. Miejsca do uzupełnienia oznaczono `...`.
Notebook otwarty z linku na karcie laboratorium nie jest zapisany na Twoim Dysku: po ponownym kliknięciu linku zobaczysz wersję wyjściową.
Na początku zajęć zapisz kopię (*Plik → Zapisz kopię na Dysku*) i pracuj na niej.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from scipy.stats import norm

rng = np.random.default_rng(2026)

## Część 1. $k$-NN ręcznie (Zadanie 3.1)

Najpierw piszemy $k$-NN sami, w numpy, i sprawdzamy wyniki z tablicy. Potem to samo liczymy biblioteką scikit-learn.

Wiersz macierzy `X_ucz` to jedna obserwacja, kolumny to cechy $x_1$, $x_2$.
Nowe punkty też zapisujemy jako macierz, z jednym wierszem na punkt, nawet jeśli punkt jest jeden.

In [ ]:
X_ucz = np.array([[1, 1], [2, 1], [4, 3], [5, 4], [2, 3], [3, 2]], dtype=float)
klasa_ucz = np.array(["A", "A", "B", "B", "B", "A"])
y_ucz = np.array([2.0, 2.5, 5.0, 6.5, 4.0, 3.5])

x_nowy = np.array([[1, 3]], dtype=float)

### 1.1. Odległości i sąsiedzi

`None` w indeksie wstawia nową oś długości 1: `X_nowe[:, None, :]` ma kształt $(m, 1, p)$, a `X_ucz[None, :, :]` -- kształt
$(1, n, p)$. Liczby są te same, zmienia się tylko kształt: tablicę dwuwymiarową zanurzamy w tablicy trójwymiarowej.
Przy odejmowaniu broadcasting rozciąga osie długości 1, więc każdy wiersz `X_nowe` zostaje zestawiony z każdym wierszem `X_ucz`.

`X_nowe[:, None, :] - X_ucz[None, :, :]` ma kształt $(m, n, p)$: element `[j, i, :]` to wektor różnic między $j$-tym
nowym punktem a $i$-tą obserwacją uczącą. Sumując po ostatniej osi (`axis=2`), dostajemy macierz odległości $m\times n$.

`np.argsort` zwraca indeksy, które ustawiają wiersz rosnąco; pierwsze $k$ z nich to $N_k(x)$.
Przy `kind="stable"` z równych odległości pierwsza jest obserwacja o niższym numerze.
Uwaga: indeksy w Pythonie zaczynają się od 0, więc obserwacja $i$ z tabeli w książce ma indeks $i-1$.

In [ ]:
def odleglosci(X_ucz, X_nowe, metryka="euklidesowa"):
    roznice = X_nowe[:, None, :] - X_ucz[None, :, :]   # kształt (m, n, p)
    if metryka == "euklidesowa":
        return ...
    if metryka == "miejska":
        return ...

def sasiedzi(X_ucz, X_nowe, k, metryka="euklidesowa"):
    D = odleglosci(X_ucz, X_nowe, metryka)
    return np.argsort(D, axis=1, kind="stable")[:, :k]   # indeksy k najbliższych, kształt (m, k)

print(np.round(odleglosci(X_ucz, x_nowy), 3))
print(sasiedzi(X_ucz, x_nowy, 1), sasiedzi(X_ucz, x_nowy, 3))

Czy zbiór $N_3(x)$ jest wyznaczony jednoznacznie? Sprawdź w wypisanych odległościach, czy trzeci i czwarty sąsiad
nie leżą w tej samej odległości.

### 1.2. Predykcje

`y_ucz[idx]` dla macierzy indeksów `idx` kształtu $(m, k)$ daje macierz wyników sąsiadów tego samego kształtu
(indeksowanie tablicą, jak w laboratorium 1). Średnia po `axis=1` to predykcja regresji dla każdego nowego punktu,
a średnia z maski `klasa_ucz[idx] == "B"` to $\widehat\eta_{\mathrm B}(x)$.

In [ ]:
def knn_regresja(X_ucz, y_ucz, X_nowe, k, metryka="euklidesowa"):
    idx = sasiedzi(X_ucz, X_nowe, k, metryka)
    return ...

def knn_eta(X_ucz, klasa_ucz, X_nowe, k, klasa, metryka="euklidesowa"):
    idx = sasiedzi(X_ucz, X_nowe, k, metryka)
    return ...   # frakcja sąsiadów z klasą `klasa`

for k in (1, 3):
    eta_B = knn_eta(X_ucz, klasa_ucz, x_nowy, k, "B")
    print(f"k = {k}:  eta_B = {eta_B},  klasa = {np.where(eta_B > 0.5, 'B', 'A')},  "
          f"regresja = {knn_regresja(X_ucz, y_ucz, x_nowy, k)}")

Punkt d): powtórz obliczenia z odległością miejską (`metryka="miejska"`). Ile obserwacji leży w odległości
równej odległości trzeciego sąsiada? Zmień kolejność wierszy w zbiorze uczącym (np. obserwację 3 na początek: `perm = [2, 0, 1, 3, 4, 5]`,
potem `X_ucz[perm]`, `klasa_ucz[perm]`) i sprawdź, czy zmienia się predykcja klasy dla $k=3$.

In [ ]:
...

### 1.3. To samo w scikit-learn

Model tworzymy z hiperparametrem $k$ (`n_neighbors`), „uczymy” metodą `fit` (dla $k$-NN to tylko zapamiętanie danych)
i pytamy o predykcję metodą `predict`. `predict_proba` zwraca $\widehat\eta$ dla klas w kolejności `klf.classes_`,
a `kneighbors` -- odległości do sąsiadów i ich indeksy. Porównaj z wynikami z 1.1 i 1.2.

In [ ]:
klf = KNeighborsClassifier(n_neighbors=3).fit(X_ucz, klasa_ucz)
reg = KNeighborsRegressor(n_neighbors=3).fit(X_ucz, y_ucz)

print(klf.kneighbors(x_nowy))
print(klf.classes_, klf.predict_proba(x_nowy), klf.predict(x_nowy))
print(reg.predict(x_nowy))

## Część 2. Rola $k$

Wracamy do przykładu z Zadania 1.14: $Y=\sin(2\pi X)+\varepsilon$, $X$ jednostajny na $[0,1]$,
$\varepsilon\sim N(0,\,0{,}3^2)$, $n=30$. Zamiast wielomianów dopasowujemy regresję $k$-NN
dla różnych $k$. Ryzyko, jak w laboratorium 1, przybliżamy średnią stratą kwadratową na dużej nowej próbie.

scikit-learn oczekuje cech w postaci macierzy $n\times p$, a tu jest jedna cecha, stąd `X.reshape(-1, 1)`.

In [ ]:
sigma = 0.3

def generuj(n):
    X = rng.uniform(0, 1, size=n)
    Y = np.sin(2 * np.pi * X) + rng.normal(0, sigma, size=n)
    return X.reshape(-1, 1), Y

X_tr, Y_tr = generuj(30)
X_te, Y_te = generuj(100_000)

In [ ]:
siatka = np.linspace(0, 1, 1000).reshape(-1, 1)
fig, osie = plt.subplots(1, 3, figsize=(11, 3.4), sharey=True)
for ax, k in zip(osie, [1, 5, 30]):
    reg = KNeighborsRegressor(n_neighbors=k).fit(X_tr, Y_tr)
    ax.scatter(X_tr, Y_tr, s=12, color="0.5")
    ax.plot(siatka, np.sin(2 * np.pi * siatka), "k--", lw=1)
    ax.plot(siatka, reg.predict(siatka), lw=1.6)
    ax.set_title(f"$k = {k}$")
plt.show()

Dla każdego $k$ oblicz błąd na zbiorze uczącym i ryzyko. Na osi poziomej wykresu jest $n/k$: im mniejsze $k$,
tym bardziej elastyczny predyktor, więc $n/k$ gra tę rolę, co stopień wielomianu w Zadaniu 1.14.

In [ ]:
def bledy_knn(X_tr, Y_tr, X_te, Y_te, wartosci_k):
    blad_uczacy, ryzyko = [], []
    for k in wartosci_k:
        reg = KNeighborsRegressor(n_neighbors=k).fit(X_tr, Y_tr)
        blad_uczacy.append(...)   # średnia strata kwadratowa na zbiorze uczącym
        ryzyko.append(...)        # średnia strata kwadratowa na nowej próbie
    return np.array(blad_uczacy), np.array(ryzyko)

n = len(Y_tr)
wartosci_k = np.array([1, 2, 3, 5, 7, 10, 15, 20, 30])
blad_uczacy, ryzyko = bledy_knn(X_tr, Y_tr, X_te, Y_te, wartosci_k)

plt.plot(n / wartosci_k, blad_uczacy, "o-", label="błąd na zbiorze uczącym")
plt.plot(n / wartosci_k, ryzyko, "o-", label="ryzyko (nowa próba)")
plt.axhline(sigma ** 2, color="gray", linestyle="--", label="błąd nieredukowalny")
plt.xscale("log")
plt.xlabel("$n/k$")
plt.ylabel("średnia strata kwadratowa")
plt.legend()
plt.show()

Pytania:

1. Dlaczego dla $k=1$ błąd na zbiorze uczącym jest równy zeru (Zadanie 1.6)? Czy mówi to coś o ryzyku?
2. Które $k$ wskazałby błąd na zbiorze uczącym, a które ryzyko?
3. Co przewiduje predyktor dla $k=n$ (Zadanie 3.3)? Ile wynosi wtedy ryzyko w przybliżeniu? (Wskazówka: $\operatorname{Var}\big(\sin(2\pi X)\big)=1/2$.)
4. Porównaj kształt obu krzywych z wykresem dla wielomianów z laboratorium 1.

Wynik zależy od wylosowanej próby uczącej. Powtórz doświadczenie dla 20 prób uczących i narysuj średnie obu krzywych.
Nową próbę możesz zmniejszyć do $10\,000$ obserwacji, żeby obliczenia trwały krócej. Następnie zmień $n$ na 300
(i wartości $k$ tak, by $k=n$ było wśród nich): czy najlepsze $k$ się zmienia? A najlepsze $n/k$?

In [ ]:
...

## Część 3. Klasy niezrównoważone (Zadanie 2.4)

Dwie klasy na prostej: klasa 1 ma rozkład $N(\mu,1)$, klasa 0 -- $N(-\mu,1)$, $\mu=1$. W przykładzie z książki
(rysunek przed zadaniem) klasy były równoliczne. Teraz klasa 1 jest rzadka: $P(Y=1)=\pi_1=0{,}1$, $P(Y=0)=\pi_0=0{,}9$.

### 3.1. Reguła Bayesa i jej błąd

Ze wzoru Bayesa $\eta(x)>1/2$ wtedy i tylko wtedy, gdy $\pi_1\,\varphi(x-\mu)>\pi_0\,\varphi(x+\mu)$, gdzie $\varphi$ to gęstość
$N(0,1)$. Po zlogarytmowaniu dostajemy próg $x^*$: reguła Bayesa wskazuje klasę 1 dla $x>x^*$. Wyznacz $x^*$ i błąd bayesowski, czyli sumę
dwóch składników: $\pi_0\,P\big(N(-\mu,1)>x^*\big)$ (klasa 0 zaklasyfikowana jako 1) i $\pi_1\,P\big(N(\mu,1)<x^*\big)$
(klasa 1 zaklasyfikowana jako 0).
Dystrybuanta $N(0,1)$ to `norm.cdf`, a $P(N(m,1)<t)$ to `norm.cdf(t - m)`.
Porównaj wynik z błędem klasyfikatora trywialnego, który zawsze wskazuje klasę 0,
i z błędem bayesowskim $\Phi(-\mu)$ dla klas równolicznych.

In [ ]:
mu, pi1 = 1.0, 0.1
pi0 = 1 - pi1

prog = ...
blad_bayesa = ...
blad_trywialny = ...
print(f"próg x* = {prog:.4f},  błąd bayesowski = {blad_bayesa:.4f},  "
      f"klasyfikator trywialny = {blad_trywialny:.4f},  klasy równoliczne = {norm.cdf(-mu):.4f}")

### 3.2. Ryzyko liczone całkowaniem

Rozkład znamy, więc ryzyko klasyfikatora możemy policzyć dokładnie, bez zbioru testowego (tak jak w kodzie rysunku
w książce): klasyfikator popełnia błąd na klasie 0 tam, gdzie wskazuje 1, i na klasie 1 tam, gdzie wskazuje 0.
Funkcja `ryzyko_dokladne` dostaje etykiety, które klasyfikator wskazuje w punktach gęstej siatki, i całkuje
gęstości ważone prawdopodobieństwami a priori po obszarach błędu. Sprawdź ją na regule Bayesa
i na klasyfikatorze trywialnym: wyniki powinny się zgadzać z 3.1.

In [ ]:
siatka = np.linspace(-7, 7, 4001)

def ryzyko_dokladne(etykiety):
    bledy_klasy_0 = np.where(etykiety == 1, pi0 * norm.pdf(siatka + mu), 0.0)
    bledy_klasy_1 = ...
    return np.trapezoid(bledy_klasy_0 + bledy_klasy_1, siatka)

print(ryzyko_dokladne((siatka > prog).astype(int)), ryzyko_dokladne(np.zeros_like(siatka)))

### 3.3. $k$-NN wobec klasyfikatora trywialnego

Losujemy próby uczące rosnącej liczności, uczymy $k$-NN z $k=\sqrt n$ (zaokrąglonym) i liczymy jego ryzyko dokładnie.
Losowanie w dwóch krokach, jak urny w laboratorium 1: najpierw klasa, potem cecha z rozkładu tej klasy.

In [ ]:
def probka(n):
    Y = (rng.random(n) < pi1).astype(int)
    X = ...   # N(mu, 1) dla klasy 1, N(-mu, 1) dla klasy 0
    return X.reshape(-1, 1), Y

licznosci = np.array([20, 50, 100, 300, 1000, 3000, 10_000])
powtorzenia = 25
bledy = np.empty((len(licznosci), powtorzenia))
for i, n in enumerate(licznosci):
    k = max(1, round(np.sqrt(n)))
    for j in range(powtorzenia):
        X, Y = probka(n)
        klf = KNeighborsClassifier(n_neighbors=k).fit(X, Y)
        bledy[i, j] = ...

srednie = bledy.mean(axis=1)
kwantyle = np.quantile(bledy, [0.1, 0.9], axis=1)
plt.errorbar(licznosci, srednie, yerr=np.abs(kwantyle - srednie), marker="o", capsize=3, label="$k$-NN, $k=\\sqrt{n}$")
plt.axhline(blad_bayesa, color="red", linestyle="--", label="błąd bayesowski")
plt.axhline(blad_trywialny, color="gray", linestyle=":", label="klasyfikator trywialny")
plt.xscale("log")
plt.xlabel("liczność próby uczącej $n$")
plt.ylabel("ryzyko")
plt.legend()
plt.show()

Pytania:

1. Od jakiego $n$ klasyfikator $k$-NN jest wyraźnie lepszy od trywialnego? Przy małym $n$ górny wąs sięga dokładnie $0{,}1$.
   Co to znaczy? Ile obserwacji klasy 1 jest średnio w próbie $n=20$, ilu sąsiadów bierze wtedy $k$-NN i ilu z nich
   musi należeć do klasy 1, żeby wskazał klasę 1?
2. Różnica między klasyfikatorem trywialnym a regułą Bayesa jest tu mała. Co ją wyznacza? Jak zmienia się, gdy $\mu$ rośnie
   albo gdy $\pi_1$ maleje? Sprawdź, zmieniając parametry w 3.1.
3. Powtórz 3.3 z $k=1$. Czy 1-NN pokonuje klasyfikator trywialny przy którymkolwiek $n$? Porównaj wynik z twierdzeniem Covera--Harta
   z wykładu: granica ryzyka 1-NN to $\mathbb E\big[2\eta(X)(1-\eta(X))\big]$, a oszacowanie z góry -- $2R^*(1-R^*)$.

In [ ]:
...